# Gold Layer: Dimension Tables

This notebook creates dimension tables for the gold layer:

* **dim_customers** - Customer dimension with standardized attributes
* **dim_products** - Product dimension with category information
* **dim_regions** - Region/country dimension for geographic analysis

In [0]:
from pyspark.sql import functions as F

CATALOG = "gwd_de"

# Source tables from silver
source_customers = f"{CATALOG}.silver.customers"
source_products = f"{CATALOG}.silver.products"
source_invoices = f"{CATALOG}.silver.invoices"

# Target dimension tables in gold
dim_customers_table = f"{CATALOG}.gold.dim_customers"
dim_products_table = f"{CATALOG}.gold.dim_products"
dim_regions_table = f"{CATALOG}.gold.dim_regions"

In [0]:
# Read from silver
df_customers = spark.table(source_customers)

# Transform to dimension
dim_customers = df_customers.select(
    F.col("customer_id"),
    F.col("customer_name"),
    F.col("customer_type"),
    F.col("country")
)

# Write to gold
dim_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(dim_customers_table)



In [0]:
# Read from silver
df_products = spark.table(source_products)

# Transform to dimension
dim_products = df_products.select(
    F.col("product_id"),
    F.col("product_name"),
    F.col("category"),
    F.col("cost_price")
)

# Write to gold
dim_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(dim_products_table)



In [0]:
# Read from silver invoices to get distinct regions
df_invoices = spark.table(source_invoices)

# Transform to dimension
dim_regions = df_invoices.select(
    F.col("region").alias("region_name"),
    F.col("region")
).distinct()

# Write to gold
dim_regions.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(dim_regions_table)
